<div
<br><br>
<h1 style="color:#7F000E;"> DMC </h1>
<h1 style="color:#7F000E;"> Aprendizaje no Supervisado y algoritmos de clusterización </h1>

<h3 style="color:#7F000E;"> Métodos No Supervisados </h3>
<h3 style="color:#7F000E;"> Algoritmo K - means </h3>
</div>
<br><br>
<div style="text-align:right">

<span style="color:#7F000E; font-size:14px;">Ing. César Quezada</span><br>
<span style="color:#7F000E; font-size:14px;">Horario: 19:00 – 22:00</span><br>
<span style="color:#7F000E; font-size:14px;">Sesión 01</span>

</div>


### Caso:
En una entidad bancaria existen varios canales de comunicación tales como: ATM, Oficinas, IVR, Banca por internet-Móvil, etc.
Sin embargo, al realizar las comunicaciones de ofertas a los clientes de dicha entidad bancaria, el cliente recibe diversas ofertas de distintos canales sin saber si les da importancia o no, por lo que ya interviene un gasto por parte de la entidad, ya que, realiza alianzas estratégicas de campaña.


Por tanto: Se requiere identificar cuál sería el medio de comunicación preferido para los clientes y así enviarles ofertas, advertencia, recordatorios, etc. más direccionadas.

### 1. Librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (7, 4)
plt.style.use('ggplot')

### 2. Extracción Base de datos

In [ ]:
dataFrame = pd.read_csv("01dataBaseMulti.txt",delimiter='|')
dataFrame.head()

In [ ]:
print("Número de filas: " + str(dataFrame.shape[0]))
print("Número de columnas: " + str(dataFrame.shape[1]))

In [ ]:
dataFrame.info()

In [ ]:
# Convertir a categórica
dataFrame['flgLimaprov'] = dataFrame['flgLimaprov'].astype('category')
# etiquetar categorías
dataFrame['flgLimaprov'] = dataFrame['flgLimaprov'].cat.rename_categories({1: 'Lima', 0: 'Provincia'})

In [ ]:
dataFrame.head()

### 3. Metodología

In [ ]:
#### 3.1 Análisis Previo (objetivo)
#### 3.2 Exploración (descriptivo, grafico barras,cajas)
#### 3.3 Transformación (standarización,cajas)
#### 3.4 Outliers (analisis y eliminación de outliers)
#### 3.5 Dimensionamiento (PCA)
#### 3.6 Modelamiento
#### 3.7 Evaluación
#### 3.8 Perfilamiento
#### 3.9 Visualización

In [ ]:
# Variables objetivo de estudio:
channelName = ['trxAplus', 'trxBcaex', 'trxSalex', 'trxBm', 'trxBxi', 'trxIvr', 'trxSbt', 'trxVent',
               'trxAtm','trxPostc', 'trxPostd']
characterName = ['edad','ingreso','sexo','flgLimaprov']

#### 3.1 Análisis Negocio

El dataset ya se encuentra trabajado a nivel de cliente con sus respectivas variables, se consideraron filtros de criterios de autoasignados, distribución histórica de transacciones, etc.

#### 3.2 Exploración

Análisis descriptivo de VARIABLES DE MODELADO

In [ ]:
dataFrame[channelName].describe()

In [ ]:
dataFrame[channelName].hist(bins = 100, figsize=(20,15))
plt.show()

In [ ]:
# Gráfico de cajas por variable en estudio:
for columnName in channelName:
    plt.title(columnName)
    plt.boxplot(dataFrame[columnName], 0, 'gD')
    plt.show()

In [ ]:
skew_channels = dataFrame[channelName].skew()
kurt_channels = dataFrame[channelName].kurtosis()

skew_channels, kurt_channels

Distribución y sparsity (clientes sin uso del canal)

In [ ]:
# Esto te dice qué canales son masivos vs nicho
# ---
zero_rate = (dataFrame[channelName] == 0).mean().sort_values(ascending=False)
zero_rate

In [ ]:
# Correlaciones >0.9, se debe considerar PCA o eliminar variables redundantes
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))
sns.heatmap(dataFrame[channelName].corr(), annot=True, cmap="coolwarm")
plt.title("Correlación entre canales")
plt.show()

Análisis descriptivo de VARIABLES DE CARACTERIZACIÓN

In [ ]:
for var in characterName:
    print(f"\n===== {var} =====")
    print(dataFrame[var].value_counts())
    print("\nProporciones:")
    print(dataFrame[var].value_counts(normalize=True))

In [ ]:
# Resumen general en una tabla
desc_char = {}

for var in characterName:
    freq = dataFrame[var].value_counts(normalize=True).reset_index()
    freq.columns = ['categoria','proporcion']
    desc_char[var] = freq

desc_char

In [ ]:
for var in characterName:
    dataFrame[var].value_counts(normalize=True).plot(kind='bar')
    plt.title(f"Distribución de {var}")
    plt.ylabel("Proporción")
    plt.show()

Relación entre variables

In [ ]:
pd.crosstab(dataFrame['edad'], dataFrame['sexo'], normalize='index')

In [ ]:
pd.crosstab(dataFrame['ingreso'], dataFrame['flgLimaprov'], normalize='index')


Perfil sociodemográfico global del cliente

In [ ]:
for var in characterName:
    print(f"\nModa de {var}: {dataFrame[var].mode()[0]}")

#### 3.3 Transformación

In [ ]:
# ------------------------------
# Creamos el objeto para escalar
# ------------------------------
from sklearn import preprocessing

#scaler = preprocessing.StandardScaler()
scaler = preprocessing.MinMaxScaler()

# Guardamos el dataframe original
df_real = dataFrame.copy()

# ************
# Lo aplicamos
# ************
for columnName in channelName:
    dataFrame[columnName] = scaler.fit_transform(dataFrame[columnName].values.reshape(-1, 1))

In [ ]:
dataFrame[channelName].head()

#### 3.4 Outliers

In [ ]:
# Cálculo de intervalo del diagrama de cajas - Método de Rango Intercuartílico
#def calculateNumOutliars(serie):
#  Q01 = serie.quantile(0.25)
#  Q03 = serie.quantile(0.75)
#  IQR = Q03 - Q01
#  a = (serie < (Q01 - 1.5 * IQR)) | (serie > (Q03 + 1.5 * IQR))
#  numOutliars = a[a == True].shape[0]
#  return numOutliars

In [ ]:
# Usamos el método de Z-score (considerando se distribuye Normalmente) --- para grandes volúmenes de datos
def calculateNumOutliars(serie):
    mu = serie.mean()
    desv = np.std(serie)
    a = ((serie-mu)/desv < -2) | ((serie-mu)/desv > 2)
    numOutliars = a[a == True].shape[0]
    return a,numOutliars

In [ ]:
numTotal = dataFrame.shape[0]
for columnName in channelName:
    a,numOutliars = calculateNumOutliars(dataFrame[columnName])
    # Creamos nuevos campos para filtrar los Outliers
    dataFrame['flg_'+columnName]=a
    print('*'+columnName)
    if numOutliars > 0:
      print("Número de valores outliars: " + str(numOutliars))
      print("Porcentaje: " + str(np.round(numOutliars * 100 / numTotal, 2)) + "%")
    else:
      print("****No hay Outliers")
    print("\n")

In [ ]:
# ************************
# Extrayendo los Outliers
# ************************
# Luego que cada variable tenga menos del 10% de Outlier, se filtra de manera Multivariada (este filtro podría ser
# considerado como un segmento Heavy)

dataFrame = dataFrame[(dataFrame['flg_trxAplus']==False)&
                      (dataFrame['flg_trxBcaex']==False)&
                      (dataFrame['flg_trxSalex']==False)&
                      (dataFrame['flg_trxBm']==False)&
                      (dataFrame['flg_trxBxi']==False)&
                      (dataFrame['flg_trxIvr']==False)&
                      (dataFrame['flg_trxSbt']==False)&
                      (dataFrame['flg_trxVent']==False)&
                      (dataFrame['flg_trxAtm']==False)&
                      (dataFrame['flg_trxPostc']==False)&
                      (dataFrame['flg_trxPostd']==False)]

# GUARDAR LOS ÍNDICES EN UNA LISTA
idx_filtrados = dataFrame.index.tolist()

print('Cantidad de Registros sin Outliers: '+str(dataFrame.shape[0]))
dataFrame[channelName].head()

#### 3.6 Modelamiento

In [ ]:
from sklearn.cluster import KMeans
from sklearn import metrics

In [ ]:
# Calculando el número de clúster adecuado:
X = dataFrame[channelName].copy()

numClus = range(1, 20)
kmeans = [KMeans(n_clusters=i,max_iter=600) for i in numClus]
kmeans
score = [kmeans[i].fit(X).score(X) for i in range(len(kmeans))]
score
plt.plot(numClus,score)
plt.xlabel('Número de Clúster')
plt.ylabel('Score')
plt.title('Curva de Inflexión')
plt.show()

In [ ]:
# Nos fijamos de los indicadores de clustering:

ctdDf = int(0.1*dataFrame.shape[0])
cluster = [kmeans[i].predict(X) for i in range(len(kmeans))]

for i in range(1,11):
    print(str(i+1)+' clústeres:')
    print('Inercia: '+str(kmeans[i].inertia_))
    print('Silueta: '+str(metrics.silhouette_score(X, cluster[i], metric='euclidean',sample_size=ctdDf)))
    print("\n")

Visualizando los grupos en 2-D para tener alguna noción de como se agrupan, en esta ocasión probaremos distintos par de variables

In [ ]:
fig = plt.figure()
f1 = dataFrame['trxAtm'].values
f2 = dataFrame['trxBm'].values

#colores=['red','green','blue','cyan','yellow']
colores=['red','green','blue']
asignar=[]
for row in cluster[1]:
    asignar.append(colores[row])

plt.scatter(f1, f2, c=asignar, s=20)
#plt.scatter(centroide[2][:, 0], centroide[2][:, 1], marker='*', c='yellow', s=100)
plt.show()

In [ ]:
fig = plt.figure()
f1 = dataFrame['trxAtm'].values
f2 = dataFrame['trxPostd'].values

#colores=['red','green','blue','cyan','yellow']
colores=['red','green','blue']
asignar=[]
for row in cluster[2]:
    asignar.append(colores[row])

plt.scatter(f1, f2, c=asignar, s=20)
#plt.scatter(centroide[2][:, 0], centroide[2][:, 1], marker='*', c='yellow', s=100)
plt.show()

#### 3.7 Evaluación

In [ ]:
numClus = [3,4,5,6]

In [ ]:
centroide = [kmeans[i].cluster_centers_ for i in range(len(kmeans))]
copy =  pd.DataFrame()

for i in numClus:
    # Distribución de los grupos por clúster:
    copy['cluster'] = cluster[i-1]
    cantidadGrupo =  pd.DataFrame()
    cantidadGrupo['ctdCliente']=copy.groupby('cluster').size()
    cantidadGrupo['pctCliente']=round(100*cantidadGrupo['ctdCliente']/cantidadGrupo['ctdCliente'].sum(),2)

    # gráfico de los grupos según su distribución:
    plt.pie(cantidadGrupo['pctCliente'], labels=cantidadGrupo.index, autopct='%1.1f%%')
    plt.title('Clúster '+str(i))
    plt.legend()
    plt.show()
    print(cantidadGrupo)
    print('\n')

In [ ]:
df_real = df_real.loc[idx_filtrados]

df = pd.DataFrame()
df = df_real.copy()

In [ ]:
nCluster = int(input('Ingrese la cantidad de cluster: '))
df['cluster'] = cluster[nCluster-1]

In [ ]:
resClus = df.groupby('cluster').agg({'cliente':'count','trxBxi':'mean','trxPostc':'mean','trxVent':'mean','trxSbt':'mean','trxAtm':'mean','trxPostd':'mean'})\
                                 .sort_values(by='cluster')
resClus = resClus.reset_index()
resClus['%'] = round(100*resClus['cliente']/df.count()[0],1)
print(f'Clientes Total: {df.count()[0]}\n')
resClus

RadarPlot

Cada cluster tendrá una forma diferente:

- Picos altos → canal dominante
- Área grande → cliente omnicanal
- Forma concentrada → mono-canal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Variables de canales (columnas del radar)
channels = ['trxBxi','trxPostc','trxVent','trxSbt','trxAtm','trxPostd']

# Indexar por cluster
radar_df = resClus.set_index('cluster')[channels]

# Normalizar (MUY IMPORTANTE para radar)
radar_norm = (radar_df - radar_df.min()) / (radar_df.max() - radar_df.min())

# Ángulos del radar
num_vars = len(channels)
angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # cerrar el círculo

# Crear gráfico polar
fig, ax = plt.subplots(figsize=(6,6), subplot_kw=dict(polar=True))

for cluster in radar_norm.index:
    values = radar_norm.loc[cluster].tolist()
    values += values[:1]

    ax.plot(angles, values, label=f'Cluster {cluster}')
    ax.fill(angles, values, alpha=0.1)

# Labels
ax.set_thetagrids(np.degrees(angles[:-1]), channels)
ax.set_title("Radar Plot de Canales por Cluster", fontsize=14)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.show()

PERFIL SOCIOECONÓMICO

In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(x='cluster', y='edad', data=df)
plt.title("Distribución Edad por Cluster")
plt.show()

plt.figure(figsize=(12,5))
sns.boxplot(x='cluster', y='ingreso', data=df)
plt.title("Distribución Ingreso por Cluster")
plt.show()

In [ ]:
sexo_cluster = pd.crosstab(df.cluster, df.sexo, normalize='index')
lima_cluster = pd.crosstab(df.cluster, df.flgLimaprov, normalize='index')

sexo_cluster.plot(kind='bar', figsize=(10,5), title="Sexo por Cluster")
plt.show()

lima_cluster.plot(kind='bar', figsize=(10,5), title="Lima vs Provincia por Cluster")
plt.show()